**■ 외부에 있는 데이터셋 다운로드**

In [3]:
# kaggle API 토큰 다운받아서 사용하는 것임!

# 1. Kaggle JSON 업로드
from google.colab import files
files.upload()  # 로컬에서 kaggle.json 파일 선택 후 업로드

# 2. kaggle.json 설정
!mkdir -p ~/.kaggle # kaggle폴더가 없으면 생성
!cp kaggle.json ~/.kaggle/ # 업로드 한 kaggle.json 파일 .kaggle 폴더로 복사
!chmod 600 ~/.kaggle/kaggle.json # kaggle.json파일 권한 설정 -> 소유자만 읽기/쓰기 가능

# ============================================================================

# 3. 데이터셋 다운로드
!kaggle datasets download -d animeshkumarnayak/pcb-fault-detection -p /content/dataset

# 4. 다운받은 데이터셋 압축 풀기
!unzip -q /content/dataset/pcb-fault-detection.zip -d /content/dataset


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/animeshkumarnayak/pcb-fault-detection
License(s): CC0-1.0
 95% 155M/163M [00:00<00:00, 1.62GB/s]
100% 163M/163M [00:00<00:00, 1.59GB/s]


**■ ultralytics 및 opencv-python 패키지 다운 받기**

ultralytics -> YOLO 학습 패키지 내장,
ultralytics에 opencv 활용할 수 있게 한번에 다운 받는 것!




In [4]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00


**■ 학습시킬 파일 위치 파악**

In [5]:
import os

dataset_dir = "/content/dataset/Data"

#  현재 파일 구조 확인
print("=== /content/dataset/Data ===")
print(os.listdir(dataset_dir))

# 학습 파일 내 data.yaml 파일 존재 여부 확인
if "data.yaml" in os.listdir(dataset_dir):
    print("✅ data.yaml 파일 발견!")
else:
    print("❌ data.yaml 파일 없음! 위치 확인 필요.")

# 학습 파일 내 train/validation/test 폴더 확인
for folder in ["train", "validation", "test"]:
    if folder in os.listdir(dataset_dir):
        print(f"✅ {folder} 폴더 존재")
    else:
        print(f"❌ {folder} 폴더 없음! 확인 필요")


=== /content/dataset/Data ===
['train', 'test', 'data.yaml', 'validation']
✅ data.yaml 파일 발견!
✅ train 폴더 존재
✅ validation 폴더 존재
✅ test 폴더 존재


■ 학습시킬 파일 개수 확인
-> images , labels 파일 개수 일치해야 함!

In [6]:
# Train
!echo "Train images:" $(ls /content/dataset/Data/train/images | wc -l)
!echo "Train labels:" $(ls /content/dataset/Data/train/labels | wc -l)

# Validation
!echo "Validation images:" $(ls /content/dataset/Data/validation/images | wc -l)
!echo "Validation labels:" $(ls /content/dataset/Data/validation/labels | wc -l)

# Test
!echo "Test images:" $(ls /content/dataset/Data/test/images | wc -l)


Train images: 1099
Train labels: 1099
Validation images: 200
Validation labels: 200
Test images: 111


■ 클래스 6번 인덱스 (resestor) 값을 가진 파일 파악

In [7]:
!grep -R "^6 " /content/dataset/Data


grep: /content/dataset/Data/train/images/VID202106071439301-78_jpg.rf.2989017482b5f18252b5d359ff2ea638.jpg: binary file matches
grep: /content/dataset/Data/train/images/VID202106071436231-12_jpg.rf.ba4dab716fdc277b15ac0f09508df562.jpg: binary file matches
grep: /content/dataset/Data/train/images/VID202106071441201-36_jpg.rf.e73c226afa8720be326455f0d39f2f33.jpg: binary file matches
grep: /content/dataset/Data/train/images/VID202106071438351-48_jpg.rf.39ea8b1da1b1ff32299e21a86d90bafd.jpg: binary file matches
grep: /content/dataset/Data/train/images/VID202106071442321-168_jpg.rf.bfad1f4677ce106fda33e6c5601dab26.jpg: binary file matches
grep: /content/dataset/Data/train/images/VID202106071441411-78_jpg.rf.74b918f68cf85f52ddd78b856dfe7832.jpg: binary file matches
grep: /content/dataset/Data/test/images/VID20210601144553-180_jpg.rf.6d11c231164cd4e7fc8533ecc3552a3c.jpg: binary file matches
grep: /content/dataset/Data/test/images/VID20210601144119-15_jpg.rf.8b7b7cdc898045b865366367e3e12095.jpg

■ 클래스 6번 인덱스 (resestor) 값을 가진 파일 삭제

In [8]:
import os

# 삭제할 라벨 파일 목록
files_to_delete = [
    "/content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.8596094c85ef4761e4b0c97a10cc5d7d.txt",
    "/content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.5fb722e549672b2766f08a1081843564.txt",
    "/content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.8a6fab79f811fa5b8039d26197e89413.txt"
]

for lbl_path in files_to_delete:
    # 삭제 전 확인
    if os.path.exists(lbl_path):
        os.remove(lbl_path)
        print(f"Deleted label: {lbl_path}")

    # 이미지 파일 경로 생성
    img_path = os.path.join(
        os.path.dirname(lbl_path).replace("labels", "images"),
        os.path.basename(lbl_path).rsplit(".", 1)[0] + ".jpg"  # 확장자가 jpg라고 가정
    )

    # 이미지 파일 삭제
    if os.path.exists(img_path):
        os.remove(img_path)
        print(f"Deleted image: {img_path}")

print("✅ 클래스 6 관련 라벨 + 매칭 이미지 삭제 완료")


Deleted label: /content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.8596094c85ef4761e4b0c97a10cc5d7d.txt
Deleted image: /content/dataset/Data/validation/images/VID202106071444271-60_jpg.rf.8596094c85ef4761e4b0c97a10cc5d7d.jpg
Deleted label: /content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.5fb722e549672b2766f08a1081843564.txt
Deleted image: /content/dataset/Data/validation/images/VID202106071444271-60_jpg.rf.5fb722e549672b2766f08a1081843564.jpg
Deleted label: /content/dataset/Data/validation/labels/VID202106071444271-60_jpg.rf.8a6fab79f811fa5b8039d26197e89413.txt
Deleted image: /content/dataset/Data/validation/images/VID202106071444271-60_jpg.rf.8a6fab79f811fa5b8039d26197e89413.jpg
✅ 클래스 6 관련 라벨 + 매칭 이미지 삭제 완료


■ 클래스 6번 인덱스 (resestor) 값을 가진 파일 삭제됐는지 다시 확인
-> 출력결과가 아무것도 안 나와야 변환된 거임

In [9]:
!grep "^6 " /content/dataset/Data/validation/labels/*.txt


■ 클래스 6번 인덱스 (resestor)파일 삭제 후 학습시킬 파일 확인

In [10]:
# Train
!echo "Train images:" $(ls /content/dataset/Data/train/images | wc -l)
!echo "Train labels:" $(ls /content/dataset/Data/train/labels | wc -l)

# Validation
!echo "Validation images:" $(ls /content/dataset/Data/validation/images | wc -l)
!echo "Validation labels:" $(ls /content/dataset/Data/validation/labels | wc -l)

# Test
!echo "Test images:" $(ls /content/dataset/Data/test/images | wc -l)


Train images: 1099
Train labels: 1099
Validation images: 197
Validation labels: 197
Test images: 111


■ 파일 내부 클래스 7번 인덱스 가지는 .txt 파일 변환
- vaildation 파일
- train 파일

클래스 7번 인덱스 -> 클래스 6번 인덱스

In [11]:
import os

# train과 validation 모두 처리
dataset_dirs = [
    "/content/dataset/Data/train/labels",
    "/content/dataset/Data/validation/labels"
]

total_modified_files = 0
total_changed_labels = 0

for labels_dir in dataset_dirs:
    if not os.path.exists(labels_dir):
        print(f"❌ 경로를 찾을 수 없습니다: {labels_dir}")
        continue

    print(f"\n 처리 중: {labels_dir}")

    modified_count = 0
    changed_labels_count = 0

    for filename in os.listdir(labels_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(labels_dir, filename)

            try:
                with open(file_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                file_changed = False

                for line in lines:
                    if line.strip():  # 빈 줄이 아닌 경우만
                        parts = line.strip().split()
                        if len(parts) > 0 and parts[0] == '7':
                            parts[0] = '6'
                            changed_labels_count += 1
                            file_changed = True
                        new_lines.append(' '.join(parts) + '\n')
                    else:
                        new_lines.append(line)  # 빈 줄 유지

                # 파일 다시 쓰기
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)

                if file_changed:
                    modified_count += 1

            except Exception as e:
                print(f" 파일 처리 중 오류 ({filename}): {e}")

    print(f"  ✅ {modified_count}개 파일 수정")
    print(f"  🔄 {changed_labels_count}개 라벨 변경 (7→6)")

    total_modified_files += modified_count
    total_changed_labels += changed_labels_count


 처리 중: /content/dataset/Data/train/labels
  ✅ 957개 파일 수정
  🔄 960개 라벨 변경 (7→6)

 처리 중: /content/dataset/Data/validation/labels
  ✅ 188개 파일 수정
  🔄 191개 라벨 변경 (7→6)


■ 파일 내부 클래스 7번 인덱스 가지는 .txt 파일 변환된건지 확인
- vaildation 파일
- traion 파일

-> 출력결과가 아무것도 안 나와야 변환된 거임 !

In [12]:
!grep "^7 " /content/dataset/Data/validation/labels/*.txt
!grep "^7 " /content/dataset/Data/train/labels/*.txt


■ 파일 내부 클래스 8번 인덱스 가지는 .txt 파일 변환
- vaildation 파일
- train 파일

클래스 8번 인덱스 -> 클래스 7번 인덱스

In [13]:
import os

# train과 validation 모두 처리
dataset_dirs = [
    "/content/dataset/Data/train/labels",
    "/content/dataset/Data/validation/labels"
]

total_modified_files = 0
total_changed_labels = 0

for labels_dir in dataset_dirs:
    if not os.path.exists(labels_dir):
        print(f"❌ 경로를 찾을 수 없습니다: {labels_dir}")
        continue

    print(f"\n 처리 중: {labels_dir}")

    modified_count = 0
    changed_labels_count = 0

    for filename in os.listdir(labels_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(labels_dir, filename)

            try:
                with open(file_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                file_changed = False

                for line in lines:
                    if line.strip():  # 빈 줄이 아닌 경우만
                        parts = line.strip().split()
                        if len(parts) > 0 and parts[0] == '8':
                            parts[0] = '7'
                            changed_labels_count += 1
                            file_changed = True
                        new_lines.append(' '.join(parts) + '\n')
                    else:
                        new_lines.append(line)  # 빈 줄 유지

                # 파일 다시 쓰기
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)

                if file_changed:
                    modified_count += 1

            except Exception as e:
                print(f" 파일 처리 중 오류 ({filename}): {e}")

    print(f"  ✅ {modified_count}개 파일 수정")
    print(f"  🔄 {changed_labels_count}개 라벨 변경 (8→7)")

    total_modified_files += modified_count
    total_changed_labels += changed_labels_count


 처리 중: /content/dataset/Data/train/labels
  ✅ 1098개 파일 수정
  🔄 2162개 라벨 변경 (8→7)

 처리 중: /content/dataset/Data/validation/labels
  ✅ 194개 파일 수정
  🔄 379개 라벨 변경 (8→7)


■ 파일 내부 클래스 8번 인덱스 가지는 .txt 파일 변환된건지 확인
- vaildation 파일
- traion 파일

-> 출력결과가 아무것도 안 나와야 변환된 거임 !

In [14]:
!grep "^8 " /content/dataset/Data/validation/labels/*.txt
!grep "^8 " /content/dataset/Data/train/labels/*.txt



■ 파일 변환된거 확인

-> 이미지와 라벨 파일명이 틀어지진 않았는지 확인하는 코드

In [15]:
import os

base_dir = "/content/dataset/Data"
folders = ["train", "validation"]

# 파일명 불일치 저장
mismatched_files = []

for folder in folders:
    img_dir = os.path.join(base_dir, folder, "images")
    lbl_dir = os.path.join(base_dir, folder, "labels")

    # 이미지 파일만 리스트
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for img_file in img_files:
        # 대응 라벨 파일 이름
        lbl_file = os.path.join(lbl_dir, img_file.rsplit(".", 1)[0] + ".txt")
        if not os.path.exists(lbl_file):
            mismatched_files.append((folder, img_file, "라벨 없음"))

    # 라벨 파일 중 이미지 없는 경우도 체크
    lbl_files = [f for f in os.listdir(lbl_dir) if f.lower().endswith(".txt")]
    for lbl_file in lbl_files:
        img_file = os.path.join(img_dir, lbl_file.rsplit(".", 1)[0] + ".jpg")
        if not os.path.exists(img_file):
            mismatched_files.append((folder, lbl_file, "이미지 없음"))

# 결과 출력
if mismatched_files:
    print("=== 파일명 불일치 / 누락 리스트 ===")
    for folder, f, reason in mismatched_files:
        print(f"[{folder}] {f} → {reason}")
else:
    print("✅ 모든 이미지와 라벨 파일명이 정상적으로 매칭됨")


✅ 모든 이미지와 라벨 파일명이 정상적으로 매칭됨


■ data.yaml 파일 수정하기

In [16]:
import os

# data.yaml 파일 경로
yaml_path = '/content/dataset/Data/data.yaml'

# 기존 파일이 있으면 삭제
if os.path.exists(yaml_path):
    os.remove(yaml_path)
    print(f" 기존 data.yaml 파일 삭제 완료: {yaml_path}")

# 새로운 data.yaml 내용 (정확한 형식)
yaml_content = """train: /content/dataset/Data/train
val: /content/dataset/Data/validation
test: /content/dataset/Data/test

nc: 8
names: ['Cap1', 'Cap2', 'Cap3', 'Cap4', 'MOSFET', 'Mov', 'Resistor', 'Transformer']
"""

# data.yaml 파일 생성
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ 새로운 data.yaml 파일 생성 완료: {yaml_path}\n")

 기존 data.yaml 파일 삭제 완료: /content/dataset/Data/data.yaml
✅ 새로운 data.yaml 파일 생성 완료: /content/dataset/Data/data.yaml



■ Google drive에 수정한 데이터셋 업로드 하기



In [18]:
import shutil
import os
from google.colab import drive

# 1. Google Drive 마운트
drive.mount('/content/drive')
print("✅ Google Drive 마운트 완료!\n")

# 2. 기존에 'colab_backup'이라는 폴더가 있으면 삭제
backup_path = '/content/drive/MyDrive/colab_backup'

if os.path.exists(backup_path):
    print(f"※ 기존 colab_backup 폴더 삭제 중... ※")
    shutil.rmtree(backup_path)
    print(f"✅ 기존 폴더 삭제 완료!\n")

# 3. 새로운 백업 폴더 생성 및 데이터셋 복사
source_path = '/content/dataset/Data'  # 현재 수정된 데이터셋 경로
destination_path = os.path.join(backup_path, 'data')

# 데이터셋 복사
shutil.copytree(source_path, destination_path)

print(f"✅ 백업 완료!")
print(f"백업 위치: {destination_path}\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive 마운트 완료!

※ 기존 colab_backup 폴더 삭제 중... ※
✅ 기존 폴더 삭제 완료!

✅ 백업 완료!
백업 위치: /content/drive/MyDrive/colab_backup/data

